# DeliveryPulse — формальная проверка гипотез

## Цель и границы

Ноутбук воспроизводит заранее зарегистрированный protocol H1–H6 на проверенном DuckDB warehouse. Данные синтетические, анализ наблюдательный, причинность не устанавливается. `p-value` не является размером эффекта; статистическая значимость рассматривается отдельно от практической.

In [ ]:
from pathlib import Path

import pandas as pd

from delivery_pulse.hypotheses import HypothesisConfig, run_hypotheses

PROJECT_ROOT = Path.cwd()
DATABASE = PROJECT_ROOT / "data" / "processed" / "delivery_pulse.duckdb"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "hypotheses"
PROTOCOL = PROJECT_ROOT / "docs" / "hypothesis_protocol.md"

if not DATABASE.is_file():
    raise FileNotFoundError("Сначала создайте проверенный DuckDB warehouse")

print(PROTOCOL.read_text(encoding="utf-8").split("## H1.", maxsplit=1)[0])

## Запуск

Pipeline повторно выполняет feasibility, но решение не использует направление эффекта или p-value. На full-наборе H1–H5 допускаются к глубокому анализу, H6 остаётся `inconclusive` и получает только заранее заданную Fisher sensitivity.

In [ ]:
run = run_hypotheses(
    HypothesisConfig(
        database=DATABASE,
        output_dir=OUTPUT_DIR,
        alpha=0.05,
        seed=42,
        min_group_size=90,
        force=True,
    )
)
print(f"Elapsed: {run.elapsed_seconds:.2f} seconds")

## Feasibility до интерпретации моделей

In [ ]:
pd.DataFrame(
    [
        item.__dict__
        if hasattr(item, "__dict__")
        else {field: getattr(item, field) for field in item.__dataclass_fields__}
        for item in run.feasibility
    ]
)

## Основные результаты и BH

`p_value_adjusted` — Benjamini–Hochberg для одного заранее определённого primary p-value на гипотезу. `not_supported` не доказывает отсутствие эффекта, а `inconclusive` не означает подтверждение или опровержение.

In [ ]:
pd.DataFrame([result.to_dict() for result in run.results])

## Диагностика и sensitivity

In [ ]:
run.diagnostics

In [ ]:
primary_terms = run.coefficients.loc[
    run.coefficients["term"].isin(
        [
            "has_loading_delay",
            "is_express",
            "has_breakdown",
            "had_scheduled_maintenance_previous_month",
        ]
    )
]
primary_terms

## Графики и отчёты

In [ ]:
for name, path in sorted(run.output_paths.items()):
    print(f"{name}: {path}")

## Ограничения

Результаты зависят от спецификации модели, контрольных переменных и синтетического сценария. Fixed effects H5 не используют иерархическое shrinkage. Редкие исходы могут давать широкие интервалы или separation. H6 не моделируется по сегментам из-за недостаточных ячеек. Автоматические бизнес-рекомендации здесь не формируются.